# CinematicEarth: 18-stop cinematic flyover for Google Earth Studio

1. Write a stop list (Markdown), optionally with camera moves per stop.
2. `cinematicearth` plans a slow, tilted 45° orbit at every city and paced hops between them, then writes **`flyover.esp`**.
3. Open it in **Earth Studio**, render, upload the render here, and get an annotated **`flyover_final.mp4`**.

In [ ]:
# 1 | install (set to the GitHub account that hosts the repo)
GITHUB_USER = "YOUR_GITHUB_USER"
!pip install -q "CinematicEarth[all] @ git+https://github.com/{GITHUB_USER}/CinematicEarth.git"
import cinematicearth as ce
print("cinematicearth", ce.__version__)

## Stops + camera moves
One line per stop: `name | lat, lon, elevation_m | moves | key=value`.
Leave moves blank for the default: `orbit 45 + zoom 7km>3.5km` over `stop_duration`.

| move | effect |
|---|---|
| `orbit 45` / `pan 45` | circle the city (negative = other way) |
| `zoom 8km>3km` / `zoom in` | pull in / push out |
| `tilt 55>70` | tilt (0 = down, 90 = horizon) |
| `truck 3km` / `flyby 3km` | sideways parallax slide past the city |
| `rise 1km` | crane up/down |
| `hold` | pause |
| `slow 1.5` / `fast 2` | stretch / shrink this stop |

`+` = moves at the same time, `;` = one after another, `@5s` = segment duration.

In [ ]:
%%writefile stops.md
---
title: 18-stop flyover
speed: slow          # glacial | slow | medium | fast  (peak on-screen speed for transits)
stop_duration: 8     # seconds circling each city
orbit: 45            # slow 45-degree orbit around each city
tilt: 60             # oblique 3D view (0 = straight down, 90 = horizon)
range_start: 7000    # m from the city when the orbit starts...
range_end: 3500      # ...pulling in to this by the end of the orbit
glide_km: 150        # shorter hops stay low and fly forward; longer hops arc up
fps: 30
# duration: 240      # uncomment to force a total length; transits are re-paced to fit
---

# Stops
Format: `name | lat, lon, ground elevation (m) | camera moves (optional) | key=value (optional)`

Leave the moves blank to get the default: `orbit 45 + zoom 7km>3.5km` over `stop_duration`.

- Coeur d'Alene, ID       | 47.6777, -116.7805, 670
- Deer Park, WA           | 47.9543, -117.4769, 645
- Winlock, WA             | 46.4910, -122.9379, 100
- Ellensburg, WA          | 46.9965, -120.5478, 470
- Portland, OR            | 45.5152, -122.6784, 50   | orbit 60 + zoom 8km>3km + tilt 58>66
- Cathlamet, WA           | 46.2032, -123.3832, 20
- McKenzie Bridge, OR     | 44.1754, -122.1606, 420
- Lyle, WA                | 45.6962, -121.2862, 60   | truck 3km + zoom 6km>4km @5s; orbit 30
- Springfield, IL         | 39.7817, -89.6501, 180
- Santa Fe, NM            | 35.6870, -105.9378, 2130
- Fort Collins, CO        | 40.5853, -105.0844, 1525
- Huntsville, AL          | 34.7304, -86.5861, 190   | orbit 60 + zoom 8km>3km; hold 1s | note=NASA EarthRISE / UAH
- Bangkok, Thailand       | 13.7563, 100.5018, 5
- Naypyidaw, Myanmar      | 19.7633, 96.0785, 115
- Kathmandu, Nepal        | 27.7172, 85.3240, 1400   | orbit 45 + tilt 55>68 + zoom 9km>4km
- Dhaka, Bangladesh       | 23.8103, 90.4125, 10
- Thimphu, Bhutan         | 27.4728, 89.6390, 2330   | orbit 45 + zoom 8km>4.5km
- Enschede, Netherlands   | 52.2215, 6.8937, 40      | orbit 60 + zoom 7km>3km; hold 1.5s

In [ ]:
# 2 | plan + write flyover.esp / flyover.kml / flyover.plan.json
plan = ce.build("stops.md", out="flyover.esp",
                 # override anything from the file here, e.g.:
                 # speed="medium", stop_duration=6, orbit=45, duration=300, cut_km=5000,
                 )

In [ ]:
# 3 | check the path: ground track, height, on-screen speed (red line = limit), pan/tilt
from cinematicearth.preview import plot_plan
plot_plan(plan);

In [ ]:
# 4 | (optional) wireframe motion check of the first ~45 s, decoded from the .esp itself
from cinematicearth.simulate import render_preview
from IPython.display import Video
render_preview(ce.build_esp(plan), plan, "wire.mp4", 640, 360, fps=15, max_seconds=45)
Video("wire.mp4", embed=True, width=640)

In [ ]:
# 5 | download the Earth Studio project (+ KML tour fallback)
from google.colab import files
files.download("flyover.esp"); files.download("flyover.kml"); files.download("flyover.plan.json")

## In Earth Studio
1. Go to [earth.google.com/studio](https://earth.google.com/studio/) and choose **File → Import**, then pick `flyover.esp`.
2. Scrub the timeline. Each city should sit at the centre of the frame while the camera circles it.
3. Choose **Render → Advanced** for full-quality JPEG frames, which download as a `.zip`. Quick render gives an `.mp4` and works too.

If Earth Studio rejects the project:
1. In Earth Studio, export any small project as `template.esp`.
2. Run `ce.build("stops.md", out="flyover.esp", template="template.esp")`.

In [ ]:
# 6 | upload the render -> annotated MP4 (pin + card per city, Google Earth credit, dips at cuts)
from cinematicearth.overlay import annotate_video, OverlayStyle, preview_frame
import matplotlib.pyplot as plt
w = plan.stop_windows()[11]
plt.figure(figsize=(12, 3.5)); plt.imshow(preview_frame(plan, w["t_start"] + 3)[..., ::-1]); plt.axis("off"); plt.show()

up = files.upload()
render = next(iter(up))
style = OverlayStyle(accent="#FFB000", show_coords=True, show_counter=True, attribution="Google Earth")
annotate_video(render, "flyover.plan.json", "flyover_final.mp4", style)
files.download("flyover_final.mp4")

## The same thing from the command line
```bash
cinematicearth build stops.md -o flyover.esp --speed slow --stop-duration 8
cinematicearth annotate render.zip flyover.plan.json -o flyover_final.mp4
```